# Module 03 — Lesson 06
# The Normal Distribution

---

> Heights. Exam scores. Measurement errors. Blood pressure. IQ scores. Manufacturing tolerances. Wildly different phenomena — yet if you plot almost any of them, you get the same shape: a symmetric bell curve, piled up in the middle, tapering off at both ends.

This is the **Normal Distribution** (also called the Gaussian distribution), and it's the single most important distribution in statistics. It shows up so often because of a remarkable fact — the **Central Limit Theorem** — which you'll prove to yourself with simulation in this lesson: when you average many independent random quantities together, the result tends toward a bell curve, *even if none of the individual quantities were bell-shaped themselves*.

This lesson ties together the z-scores from Lesson 02, the percentiles from Lesson 03, and the random variables from Lesson 05 into one complete picture.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Explain why the Normal distribution appears so often, using the Central Limit Theorem
- Compute the Normal PDF from scratch and understand its two parameters (mean, std dev)
- Standardize any Normal variable to the Standard Normal using z-scores
- Compute probabilities (areas under the curve) using the CDF, both numerically and via the exact formula
- Rigorously derive the 68-95-99.7 empirical rule for a true Normal distribution
- Find percentiles of a Normal distribution using the inverse CDF
- Demonstrate the Central Limit Theorem yourself via simulation

---
## 1. The Normal Distribution — Parameters and Shape

The Normal distribution is fully described by just two parameters:

$$f(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

- **μ (mu)** — the mean, where the peak sits
- **σ (sigma)** — the standard deviation, how wide/narrow the curve is

Notation: $X \sim N(\mu, \sigma^2)$ means "X follows a Normal distribution with mean μ and variance σ²."

In [ ]:
import math

def normal_pdf(x, mu, sigma):
    '''Normal probability density function at point x.'''
    coeff = 1 / (sigma * math.sqrt(2 * math.pi))
    exponent = -((x - mu) ** 2) / (2 * sigma ** 2)
    return coeff * math.exp(exponent)


def ascii_curve(pdf_func, mu, sigma, width=60, height=12, spread=4):
    '''Draw a rough ASCII plot of a PDF curve.'''
    lo, hi = mu - spread * sigma, mu + spread * sigma
    xs = [lo + i * (hi - lo) / (width - 1) for i in range(width)]
    ys = [pdf_func(x, mu, sigma) for x in xs]
    y_max = max(ys)

    grid = [[" "] * width for _ in range(height)]
    for col, y in enumerate(ys):
        row = height - 1 - int((y / y_max) * (height - 1))
        grid[row][col] = "*"

    for row in grid:
        print("  |" + "".join(row) + "|")
    print(f"   {lo:.0f}{' '*(width//2 - 6)}mu={mu:.0f}{' '*(width//2 - 6)}{hi:.0f}")


print("Exam scores ~ N(mu=70, sigma=10):")
ascii_curve(normal_pdf, mu=70, sigma=10)
print()
print("Same mean, narrower spread (sigma=4):")
ascii_curve(normal_pdf, mu=70, sigma=4)

---
## 2. The Standard Normal Distribution

The **Standard Normal** is the special case $Z \sim N(0, 1)$ — mean 0, std dev 1. Any Normal variable can be converted to it using the **z-score** formula from Lesson 02:

$$z = \frac{x - \mu}{\sigma}$$

This matters because it lets you compare values from *completely different* Normal distributions on the same scale.

In [ ]:
# Two different exams, different scales — whose performance is more impressive?
# Exam A: mu=70, sigma=10.  Priya scored 85.
# Exam B: mu=500, sigma=100. Rohan scored 650.

priya_score, mu_a, sigma_a = 88, 70, 10
rohan_score, mu_b, sigma_b = 650, 500, 100

z_priya = (priya_score - mu_a) / sigma_a
z_rohan = (rohan_score - mu_b) / sigma_b

print(f"Priya: score={priya_score}, exam ~ N({mu_a}, {sigma_a}) -> z = {z_priya:+.2f}")
print(f"Rohan: score={rohan_score}, exam ~ N({mu_b}, {sigma_b}) -> z = {z_rohan:+.2f}")
print()
if z_priya > z_rohan:
    print("Priya performed relatively better, despite the lower raw score —")
    print("her z-score is higher once both exams are placed on the same scale.")
else:
    print("Rohan performed relatively better once both exams are on the same scale.")

---
## 3. Computing Probabilities — the CDF

The **Cumulative Distribution Function (CDF)** gives $P(X \leq x)$ — the area under the PDF curve up to x. We can approximate this numerically (fine-grid summation, same technique as Lesson 05), or compute it *exactly* using Python's built-in error function `math.erf`.

$$CDF(x; \mu, \sigma) = \frac{1}{2}\left[1 + \text{erf}\left(\frac{x-\mu}{\sigma\sqrt{2}}\right)\right]$$

In [ ]:
def cdf_numeric(x, mu, sigma, dx=0.001):
    '''Approximate P(X <= x) by summing pdf * dx from far left up to x.'''
    lo = mu - 8 * sigma
    total = 0.0
    point = lo
    while point < x:
        total += normal_pdf(point, mu, sigma) * dx
        point += dx
    return total


def cdf_exact(x, mu, sigma):
    '''Exact P(X <= x) using the error function.'''
    return 0.5 * (1 + math.erf((x - mu) / (sigma * math.sqrt(2))))


mu, sigma = 70, 10
x = 85

p_numeric = cdf_numeric(x, mu, sigma)
p_exact = cdf_exact(x, mu, sigma)

print(f"P(X <= {x}) for X ~ N({mu}, {sigma}):")
print(f"  Numerical integration (Riemann sum) : {p_numeric:.4f}")
print(f"  Exact formula (math.erf)             : {p_exact:.4f}")
print()
print("Both agree — the numerical version builds intuition (it's literally")
print("summing tiny slices of area), the exact formula is what you'd use")
print("in practice (faster, no approximation error).")
print()

# P(a <= X <= b) = CDF(b) - CDF(a)
p_between = cdf_exact(80, mu, sigma) - cdf_exact(60, mu, sigma)
print(f"P(60 <= X <= 80) = CDF(80) - CDF(60) = {p_between:.4f}")

---
## 4. The 68-95-99.7 Rule — Rigorously Derived

Lesson 02 introduced this rule by checking it *empirically* on a small sample. Now that we have an exact CDF, we can derive it *exactly* for a true Normal distribution.

In [ ]:
mu, sigma = 70, 10

print(f"For X ~ N({mu}, {sigma}):")
print(f"  {'Range':<20} {'P(within range)':>18}")
print("  " + "-" * 40)
for k in [1, 2, 3]:
    p_within = cdf_exact(mu + k * sigma, mu, sigma) - cdf_exact(mu - k * sigma, mu, sigma)
    print(f"  mu +/- {k}*sigma{'':<9} {p_within:>17.4%}")

print()
print("These match 68.27%, 95.45%, 99.73% EXACTLY — the empirical rule isn't")
print("just a rough guideline, it's a precise mathematical property of every")
print("true Normal distribution, regardless of its mu and sigma.")

---
## 5. Verification Against Python's Standard Library

Python's `statistics` module (3.8+) includes `NormalDist`, which computes exactly what we just built by hand.

In [ ]:
from statistics import NormalDist

mu, sigma, x = 70, 10, 85
nd = NormalDist(mu, sigma)

print(f"{'Function':<20} {'Ours':>12} {'stdlib':>12}  Match?")
print("-" * 52)

checks = [
    ("pdf(x)",  normal_pdf(x, mu, sigma), nd.pdf(x)),
    ("cdf(x)",  cdf_exact(x, mu, sigma),  nd.cdf(x)),
]
for label, ours, lib in checks:
    match = abs(ours - lib) < 1e-9
    print(f"{label:<20} {ours:>12.6f} {lib:>12.6f}  {'match' if match else 'MISMATCH'}")

---
## 6. Percentiles via the Inverse CDF

The **inverse CDF** (quantile function) answers the reverse question: "what value x has exactly p probability below it?" This ties directly back to the percentiles from Lesson 03. We can find it with binary search over our own `cdf_exact()`, since CDF is always increasing.

In [ ]:
def inverse_cdf(p, mu, sigma, tol=1e-6):
    '''Binary search for the x where cdf_exact(x, mu, sigma) == p.'''
    lo, hi = mu - 10 * sigma, mu + 10 * sigma
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if cdf_exact(mid, mu, sigma) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


# A national exam: scores ~ N(500, 100), similar to SAT-style scoring
mu, sigma = 500, 100

score_90th = inverse_cdf(0.90, mu, sigma)
stdlib_90th = NormalDist(mu, sigma).inv_cdf(0.90)

print(f"Score at the 90th percentile:")
print(f"  Ours   : {score_90th:.1f}")
print(f"  stdlib : {stdlib_90th:.1f}")
print()

# What fraction score above 650?
p_above_650 = 1 - cdf_exact(650, mu, sigma)
print(f"P(score > 650) = 1 - CDF(650) = {p_above_650:.4f}  (~{p_above_650*100:.1f}% of test-takers)")

---
## 7. The Central Limit Theorem — Proving It With Simulation

**The Central Limit Theorem (CLT):** if you take repeated random samples from *any* distribution (even a badly skewed one) and compute each sample's mean, the distribution of those sample means approaches Normal as the sample size grows — regardless of the original distribution's shape.

In [ ]:
import random
random.seed(11)

def skewed_sample():
    '''A deliberately NON-normal, right-skewed random variable (values in [0, 1]).'''
    return random.random() ** 3   # cubing skews values heavily toward 0

def mean(data):
    return sum(data) / len(data)

def ascii_histogram(data, bins=16, width=40, label=""):
    lo, hi = min(data), max(data)
    bin_width = (hi - lo) / bins
    counts = [0] * bins
    for v in data:
        idx = min(int((v - lo) / bin_width), bins - 1)
        counts[idx] += 1
    max_count = max(counts)

    print(f"  {label}")
    for i, c in enumerate(counts):
        bar_len = int(c / max_count * width) if max_count else 0
        print(f"    {lo + i*bin_width:6.3f} | {'#' * bar_len}")
    print()


# The raw variable itself: heavily skewed, NOT bell-shaped
single_draws = [skewed_sample() for _ in range(5000)]
ascii_histogram(single_draws, label="Original skewed_sample() — clearly NOT normal")

# Now look at the distribution of SAMPLE MEANS for growing sample sizes
def sample_means(n_trials, sample_size):
    return [mean([skewed_sample() for _ in range(sample_size)]) for _ in range(n_trials)]

means_n2  = sample_means(2000, 2)
means_n30 = sample_means(2000, 30)

ascii_histogram(means_n2,  label="Distribution of sample means (sample size n=2)")
ascii_histogram(means_n30, label="Distribution of sample means (sample size n=30) -- looks bell-shaped!")

In [ ]:
def std_dev(data):
    m = mean(data)
    return math.sqrt(sum((x - m) ** 2 for x in data) / (len(data) - 1))

pop_std = std_dev(single_draws)

print("The Central Limit Theorem also predicts EXACTLY how much narrower the")
print("distribution of sample means gets: std(sample mean) = population_std / sqrt(n)")
print()
print(f"{'Sample size n':<15} {'Predicted std':>15} {'Observed std':>15}")
print("-" * 47)
for n, means in [(2, means_n2), (30, means_n30)]:
    predicted = pop_std / math.sqrt(n)
    observed = std_dev(means)
    print(f"{n:<15} {predicted:>15.4f} {observed:>15.4f}")

print()
print("Bigger samples -> sample means cluster more tightly around the true")
print("population mean, AND the shape becomes more normal. This is why so many")
print("statistical methods (confidence intervals, hypothesis tests, A/B testing)")
print("can assume normality even when the underlying data isn't normal at all —")
print("they operate on SAMPLE MEANS, not raw data.")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Assuming a dataset is Normal without checking -----------------

# Household income is famously right-skewed (a few very high earners pull the mean up)
incomes = [25000, 28000, 30000, 32000, 33000, 35000, 36000, 38000,
           40000, 42000, 45000, 50000, 480000]  # one very high earner

def median(data):
    s = sorted(data)
    n = len(s)
    return s[n//2] if n % 2 else (s[n//2-1] + s[n//2]) / 2

income_mean = mean(incomes)
income_median = median(incomes)

print("Mistake 1 — Assuming data is Normal without checking:")
print(f"  Mean income   : {income_mean:,.0f}")
print(f"  Median income : {income_median:,.0f}")
print()
print("  In a TRUE Normal distribution, mean == median (perfect symmetry).")
print(f"  Here they differ by {abs(income_mean - income_median):,.0f} — a big gap signals")
print("  skew. Applying the 68-95-99.7 rule or z-score outlier detection to")
print("  skewed data like this will give misleading results.")

In [ ]:
# -- Mistake 2: Confusing PDF height with probability -------------------------

narrow_density = normal_pdf(70, mu=70, sigma=0.1)   # very narrow curve

print("Mistake 2 — A PDF value is NOT a probability:")
print(f"  normal_pdf(70, mu=70, sigma=0.1) = {narrow_density:.4f}")
print()
print("  This EXCEEDS a value you might expect a 'probability' to have — but")
print("  it's fine, because it's a DENSITY, not a probability. Only the AREA")
print("  under the curve (from the CDF) between two points is a probability.")

In [ ]:
# -- Mistake 3: Forgetting P(X > x) = 1 - CDF(x) -------------------------------

mu, sigma, x = 70, 10, 85

wrong = cdf_exact(x, mu, sigma)          # WRONG if you want P(X > x)
correct = 1 - cdf_exact(x, mu, sigma)    # CORRECT

print("Mistake 3 — Sign error: forgetting to subtract from 1:")
print(f"  cdf_exact({x}) alone       = {wrong:.4f}   <- this is P(X <= {x}), not P(X > {x})")
print(f"  1 - cdf_exact({x})         = {correct:.4f}   <- this IS P(X > {x})")
print()
print("  The CDF always answers 'less than or equal to.' For 'greater than,'")
print("  you must subtract from 1. This is one of the most common bugs when")
print("  translating a word problem into code.")

In [ ]:
# -- Mistake 4: Wrong order when computing P(a <= X <= b) ---------------------

mu, sigma = 70, 10
a, b = 60, 80

wrong = cdf_exact(a, mu, sigma) - cdf_exact(b, mu, sigma)     # backwards! negative
correct = cdf_exact(b, mu, sigma) - cdf_exact(a, mu, sigma)   # CDF(upper) - CDF(lower)

print("Mistake 4 — Subtracting CDFs in the wrong order:")
print(f"  WRONG:   CDF(a) - CDF(b) = {wrong:.4f}   <- negative, clearly a bug")
print(f"  CORRECT: CDF(b) - CDF(a) = {correct:.4f}")
print()
print("  Probability can never be negative — a negative result is an instant")
print("  sign that the upper and lower bounds got swapped.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: PDF and ASCII Curve

A machine part's diameter is Normally distributed with mu=25mm, sigma=0.5mm. Compute `normal_pdf(25, 25, 0.5)` (the peak density) and use `ascii_curve()` to plot the distribution.

In [ ]:
# Your code here

### Exercise 2 — Probability Between Two Values

Using the same part (mu=25, sigma=0.5), compute the probability a randomly selected part has a diameter between 24.5mm and 25.5mm using `cdf_exact()`.

In [ ]:
# Your code here

### Exercise 3 — Finding a Percentile

A company's employee salaries are approximately Normal with mu=₹600,000 and sigma=₹120,000. Using `inverse_cdf()`, find the salary at the 75th percentile. Then verify your answer against `statistics.NormalDist(600000, 120000).inv_cdf(0.75)`.

In [ ]:
from statistics import NormalDist
# Your code here

### Exercise 4 — CLT With a Different Skewed Distribution

Write your own skewed random variable (for example, `random.random() ** 5`, or `random.expovariate(1)`), then reuse `sample_means()` and `ascii_histogram()` from Section 7 to show that its sample means (n=30) look approximately Normal, even though the raw variable does not.

In [ ]:
import random
# Your code here

### Exercise 5 — (Challenge) Six Sigma Quality Control

A manufacturing process produces bolts with diameter ~ N(mu=10mm, sigma=0.02mm). The specification limits are 9.94mm to 10.06mm — any bolt outside this range is defective.

1. Compute the probability a bolt is defective (hint: `P(defective) = 1 - P(9.94 <= X <= 10.06)`)
2. Compute how many defective bolts you'd expect out of 1,000,000 produced
3. If the process std dev could be reduced to 0.015mm (tighter control), recompute the defect rate — by how much does it improve?

In [ ]:
mu = 10
lower_spec, upper_spec = 9.94, 10.06
# Your code here

---
## 📌 Key Takeaways

- **The Normal distribution is defined by just two parameters — mean (μ) and standard deviation (σ) — and every z-score, percentile, and empirical-rule calculation from Lessons 02–03 is really Normal-distribution math underneath.** Standardizing with `z = (x - μ) / σ` converts any Normal variable to the Standard Normal, letting you compare values across completely different scales.

- **The CDF answers "what fraction of values fall below x," and `math.erf` gives you an exact, fast formula for it** — no need for numerical integration in practice, though building `cdf_numeric()` from scratch is what makes the "area under the curve" idea concrete. Always double-check direction (`P(X > x) = 1 - CDF(x)`) and order (`CDF(b) - CDF(a)` for `a < b`).

- **The Central Limit Theorem is why the Normal distribution shows up everywhere, even when raw data doesn't look Normal at all.** Averaging many independent samples pulls the distribution of those averages toward a bell curve — which is the mathematical foundation behind confidence intervals, hypothesis testing, and A/B testing throughout the rest of this course.

---
## What's Next?

**Lesson 07 — Scalars and Vectors**
You've now covered the full Statistics and Probability toolkit for this module. Next, the course shifts to **Linear Algebra** — starting with vectors, the native language every dataset gets translated into before a machine learning model can touch it. A single row of your data (like one student's age, score, and attendance) is really just a vector in disguise.